In [2]:
import os, glob, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cfgrib
import xarray as xr

In [ ]:
path          = 'C:\\Users\\maili\\Desktop\\KU\\AppMachineLearning\\Final_Project\\data\\'
MODEL_PATH    = 'saved_models/best_precip_model_2015_2026.pt'
LAG           = 24
HORIZON       = 24
FINAL_EPOCHS  = 3
BATCH         =  512 # 128
SEED          = 42
RAIN_THRESH   = 0.1   # mm — threshold for "raining"
TARGET_HOUR   = 17    # predict precipitation at 17:00 next day
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [6]:
# load in ds0 and ds1 for all years, then concatenate along time dimension
ds0 = xr.open_dataset("data\\ds0_all_years.nc")
ds1 = xr.open_dataset("data\\ds1_all_years.nc")

In [7]:

# ── 2. FLATTEN ds1 (time, step) → valid_time ─────────────────────────────────
tp_flat   = ds1["tp"].stack(alltime=("time", "step"))
ssrd_flat = ds1["ssrd"].stack(alltime=("time", "step"))

vt_index     = pd.DatetimeIndex(tp_flat.valid_time.values)
tp_flat_np   = np.moveaxis(tp_flat.values,   -1, 0)   # (96768, 5, 5)
ssrd_flat_np = np.moveaxis(ssrd_flat.values, -1, 0)

# deduplicate by valid_time
sort_idx  = np.argsort(vt_index)
vt_sorted = vt_index[sort_idx]
tp_s      = tp_flat_np[sort_idx]
ssrd_s    = ssrd_flat_np[sort_idx]

_, first_occ = np.unique(vt_sorted, return_index=True)
vt_u   = vt_sorted[first_occ]
tp_u   = tp_s[first_occ]
ssrd_u = ssrd_s[first_occ]

# align to ds0 hourly grid
ds0_idx = pd.DatetimeIndex(ds0.time.values)
loc     = pd.DatetimeIndex(vt_u).get_indexer(ds0_idx)
valid   = loc != -1
print(f"Unmatched ds0 times: {(loc==-1).sum()}")

tp_aligned   = tp_u[loc[valid]]      # (96624, 5, 5)
ssrd_aligned = ssrd_u[loc[valid]]
ds0_valid    = {v: ds0[v].values[valid] for v in ['sp','tcc','u10','v10','t2m']}
ds0_time     = ds0_idx[valid]
print(f"Aligned: {len(ds0_time)} timesteps")

Unmatched ds0 times: 0
Aligned: 96624 timesteps


In [8]:
tp_lookup = {t: i for i, t in enumerate(ds0_time)}

target_list, input_mask = [], []
for i, t in enumerate(ds0_time):
    t_pd = pd.Timestamp(t)
    next_17 = (t_pd + pd.Timedelta(days=1)).normalize().replace(hour=TARGET_HOUR)
    if next_17 in tp_lookup:
        j = tp_lookup[next_17]
        target_list.append(tp_aligned[j])   # (5, 5) in metres
        input_mask.append(i)

input_mask  = np.array(input_mask)
y_raw_mm    = np.stack(target_list).astype(np.float32) * 1000  # → mm, (N, 5, 5)
print(f"Samples with valid next-day 17:00 target: {len(input_mask)}")
print(f"Target tp range: {y_raw_mm.min():.4f} – {y_raw_mm.max():.4f} mm")


Samples with valid next-day 17:00 target: 96576
Target tp range: 0.0000 – 12.8016 mm


In [9]:
# ── 4. BUILD INPUT CHANNELS ───────────────────────────────────────────────────
channels, chan_names = [], []
for var in ['sp', 'tcc', 'u10', 'v10', 't2m']:
    channels.append(ds0_valid[var][input_mask].astype(np.float32))
    chan_names.append(var)
channels.append(tp_aligned[input_mask].astype(np.float32) * 1000)   # mm
chan_names.append("tp")
channels.append(ssrd_aligned[input_mask].astype(np.float32))
chan_names.append("ssrd")

ref_time = ds0_time[input_mask]
N_FEATURES = len(channels)

print(f"\nChannels ({N_FEATURES}):")
for name, ch in zip(chan_names, channels):
    print(f"  {name}: {ch.shape}")
print(f"Time range: {ref_time[0]} → {ref_time[-1]}")



Channels (7):
  sp: (96576, 5, 5)
  tcc: (96576, 5, 5)
  u10: (96576, 5, 5)
  v10: (96576, 5, 5)
  t2m: (96576, 5, 5)
  tp: (96576, 5, 5)
  ssrd: (96576, 5, 5)
Time range: 2015-01-01 00:00:00 → 2026-04-15 23:00:00


In [10]:
# ── 5. LOG TRANSFORM & STACK ──────────────────────────────────────────────────
X_raw = np.stack(channels, axis=1).astype(np.float32)  # (N, C, 5, 5)
y_raw = np.log1p(y_raw_mm)                              # (N, 5, 5)

tp_idx = chan_names.index("tp")
X_raw[:, tp_idx] = np.log1p(X_raw[:, tp_idx])

ssrd_idx = chan_names.index("ssrd")
# ssrd in J/m² — normalise but no log needed
ssrd_mean = X_raw[:, ssrd_idx].mean()
ssrd_std  = X_raw[:, ssrd_idx].std() + 1e-8
X_raw[:, ssrd_idx] = (X_raw[:, ssrd_idx] - ssrd_mean) / ssrd_std

N = X_raw.shape[0]
print(f"\nX_raw: {X_raw.shape} | y_raw: {y_raw.shape}")


X_raw: (96576, 7, 5, 5) | y_raw: (96576, 5, 5)


In [11]:
# ── 6. SLIDING WINDOWS ────────────────────────────────────────────────────────
def make_windows(X, y, lag=LAG):
    # each window: X[i-lag:i] → y[i]  (y[i] is already next-day-17:00 target)
    idx = np.arange(lag, len(X))
    Xw  = np.stack([X[i-lag:i] for i in idx]).astype(np.float32)  # (N, lag, C, H, W)
    yw  = y[idx].astype(np.float32)                                # (N, H, W)
    return Xw, yw

X_all_w, y_all = make_windows(X_raw, y_raw)
print(f"Windows: X={X_all_w.shape}, y={y_all.shape}")

train_end = int(0.70 * len(X_all_w))
val_end   = int(0.85 * len(X_all_w))
X_train_w, y_train = X_all_w[:train_end],          y_all[:train_end]
X_val_w,   y_val   = X_all_w[train_end:val_end],   y_all[train_end:val_end]
X_test_w,  y_test  = X_all_w[val_end:],            y_all[val_end:]


Windows: X=(96552, 24, 7, 5, 5), y=(96552, 5, 5)


In [12]:
# ── 7. NORMALISE INPUTS ───────────────────────────────────────────────────────
def fit_scale(Xw):
    mean = Xw.mean(axis=(0, 1, 3, 4))
    std  = Xw.std( axis=(0, 1, 3, 4)) + 1e-8
    return mean, std

def transform(Xw, sc):
    m, s = sc
    return ((Xw - m[None,None,:,None,None]) /
                  s[None,None,:,None,None]).astype(np.float32)

scaler  = fit_scale(X_train_w)
X_tr_sc = transform(X_train_w, scaler)
X_va_sc = transform(X_val_w,   scaler)
X_te_sc = transform(X_test_w,  scaler)
y_tr, y_va, y_te = y_train, y_val, y_test

In [13]:

# ── 8. DATASET & LOADER ───────────────────────────────────────────────────────
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def loader(X, y, shuffle):
    return DataLoader(SeqDS(X, y), batch_size=BATCH, shuffle=shuffle,
                      num_workers=0, pin_memory=False)


In [20]:
# ── 9. MODEL ──────────────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden):
        super().__init__()
        self.conv = nn.Conv2d(in_ch + hidden, 4 * hidden, 3, padding=1)
        self.hidden = hidden
    def forward(self, x, h, c):
        gates = self.conv(torch.cat([x, h], dim=1))
        i, f, g, o = gates.chunk(4, dim=1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c
    
class ConvLSTMModel(nn.Module):
    def __init__(self, in_ch, hidden=64, dropout=0.2):
        super().__init__()
        self.hidden = hidden
        self.cell   = ConvLSTMCell(in_ch, hidden)
        self.norm   = nn.GroupNorm(8, hidden)
        self.drop   = nn.Dropout2d(dropout)
        self.backbone = nn.Sequential(
            nn.Conv2d(hidden,        hidden,      3, padding=1), nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(hidden,        hidden // 2, 3, padding=1), nn.ReLU(),
        )
        self.prob_head   = nn.Conv2d(hidden // 2, 1, 1)
        self.amount_head = nn.Conv2d(hidden // 2, 1, 1)

    def forward(self, x):
        B, T, C, H, W = x.shape
        h = torch.zeros(B, self.hidden, H, W, device=x.device)
        c = torch.zeros(B, self.hidden, H, W, device=x.device)
        for t in range(T):
            h, c = self.cell(x[:, t], h, c)
        h    = self.drop(self.norm(h))
        feat = self.backbone(h)
        prob   = self.prob_head(feat).squeeze(1)    # raw logits, NO sigmoid
        amount = self.amount_head(feat).squeeze(1)
        return prob, amount

In [19]:

def rain_loss(prob_logits, amount, target):
    rain_mask = (target > np.log1p(RAIN_THRESH)).float()
    bce       = nn.functional.binary_cross_entropy_with_logits(prob_logits, rain_mask)
    weight    = 1.0 + 20.0 * rain_mask
    mse       = (weight * (amount - target) ** 2).mean()
    mae       = (weight * torch.abs(amount - target)).mean()
    amt_loss  = 0.7 * mse + 0.3 * mae
    return bce + amt_loss, bce, amt_loss


In [ ]:
best_hidden  = 32
best_lr      = 1e-3
best_dropout = 0.1

model     = ConvLSTMModel(N_FEATURES, best_hidden, best_dropout).to(device)
opt       = torch.optim.Adam(model.parameters(), lr=best_lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min',
                                                        factor=0.5, patience=3)
tr_loader = loader(X_tr_sc, y_tr, True)
va_loader = loader(X_va_sc, y_va, False)

best_val, best_state = float("inf"), None
train_losses, val_losses = [], []
PATIENCE, patience_counter = 3, 0

os.makedirs("saved_models", exist_ok=True)

for epoch in range(FINAL_EPOCHS):
    model.train()
    tl = []
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        prob, amount = model(xb)
        loss, bce, amt_loss = rain_loss(prob, amount, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl.append(loss.item())
    train_losses.append(np.mean(tl))

    model.eval()
    vl = []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb, yb = xb.to(device), yb.to(device)
            l, _, _ = rain_loss(*model(xb), yb)
            vl.append(l.item())
    val = np.mean(vl)
    val_losses.append(val)
    scheduler.step(val)

    print(f"Epoch {epoch+1:02d}: train={train_losses[-1]:.5f}, val={val:.5f}")

    if val < best_val:
        best_val   = val
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, MODEL_PATH)
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
print(f"\nLoaded best model — val loss: {best_val:.5f}")

NameError: name 'self' is not defined